# RF-DETR COCO Accuracy: Reproduce the Pretrained Numbers

Score every current pretrained RF-DETR checkpoint on the full COCO val2017 split (5000 images), at batch 1, through `RFDETRInference.predict(image, threshold=0.001, include_source_image=False)`. This is the same public call users make, with default preprocessing and a low score floor that preserves the precision-recall curve.

- Detection reports box AP and AP50.
- Segmentation reports box and mask AP and AP50.
- Keypoint Preview reports box and keypoint AP and AP50.

AP means COCO AP50:95 unless explicitly labelled AP50. Saved scores are fractions; the final comparison table uses percentages. The Benchmarks page uses a different harness (SAB: T4, TensorRT 10.4, FP16), so small differences are expected. The final table prints the PyTorch difference against SAB in percentage points.

## Install and import

The train extra supplies faster-coco-eval, which the shared evaluator needs even though this notebook never trains a model. Add the optional extras only for the models and arms you select. Use a checkout containing `evaluate_coco_ap`; the private benchmark helpers are reproducibility utilities rather than a supported library API. Restart the kernel after installing so version provenance describes the packages actually in use.

In [ ]:
# Install evaluation dependencies before importing the scoring pipeline.
!pip install "rfdetr[train]"
!pip install "rfdetr[plus]" "rfdetr_plus>=1.2.0"  # Optional: all five Plus sizes.
!pip install "rfdetr[tensorrt]"  # NVIDIA CUDA export and inference for the TensorRT arms.

Keep imports together so every dependency of the scoring path is visible in one place.

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import time
from functools import partial
from pathlib import Path
from typing import Any
from urllib.parse import quote

import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm

import rfdetr
from rfdetr import RFDETR, RFDETRInference
from rfdetr.datasets._develop import _download_and_extract
from rfdetr.export._benchmark import _download, evaluate_coco_ap, fetch_coco_val2017

## Select the checkpoints and runtimes

The defaults cover every non-Plus checkpoint and every arm: eager PyTorch FP32, and static TensorRT FP16 and INT8 engines, so an NVIDIA GPU is required. On another device, set `ARMS = ("pytorch-fp32",)`. Full validation is the publication path; a seeded subset is useful for checking the pipeline before paying for all model exports and 5000 predictions per arm.

> **Note:** PML 1.0 covers the Plus detection models (Atto, Femto, Pico, XLarge and 2XLarge): commercial use is included when you deploy on Roboflow, and commercial self-hosting requires the Enterprise add-on. Apache 2.0 covers the package, Nano–Large detection, every Seg size and Keypoint: commercial, closed-source and self-hosted use is included without a license fee or an obligation to open-source your code. Set `ACCEPT_PLATFORM_MODEL_LICENSE = True` only after accepting the Plus model terms.

In [ ]:
PLUS_MODELS = ("RFDETRAtto", "RFDETRFemto", "RFDETRPico", "RFDETRXLarge", "RFDETR2XLarge")
MODELS = (
    "RFDETRNano",
    "RFDETRSmall",
    "RFDETRMedium",
    "RFDETRLarge",
    "RFDETRSegNano",
    "RFDETRSegSmall",
    "RFDETRSegMedium",
    "RFDETRSegLarge",
    "RFDETRSegXLarge",
    "RFDETRSeg2XLarge",
    "RFDETRKeypointPreview",
)  # add `+ PLUS_MODELS` after accepting PML 1.0 below
ACCEPT_PLATFORM_MODEL_LICENSE = False
ARMS = ("pytorch-fp32", "tensorrt-fp16", "tensorrt-int8")
FULL_VAL = True
N_EVAL = 500
SEED = 0
COCO_ROOT = Path("coco")
WORK_DIR = Path("coco_accuracy")
RESULTS_PATH = WORK_DIR / "results.jsonl"
N_CALIB = 128

Reject missing license acceptance and unsupported runtime choices before downloading data or weights. Model classes resolve through `rfdetr` later, so Plus imports stay lazy and an unknown name fails there.

In [ ]:
assert MODELS, "Select at least one model."
assert ACCEPT_PLATFORM_MODEL_LICENSE or not set(MODELS) & set(PLUS_MODELS), (
    "Plus models require ACCEPT_PLATFORM_MODEL_LICENSE = True after accepting PML 1.0."
)
assert torch.cuda.is_available() or ARMS == ("pytorch-fp32",), "TensorRT arms require CUDA."
assert N_EVAL > 0 and N_CALIB > 0, "Evaluation and calibration image counts must be positive."

## Prepare COCO val2017

Full validation downloads the complete val2017 image archive first. This avoids 5000 individual transfers, which can hit socket timeouts. The completion check requires 5000 nonempty JPEG files. The shared subset fetch then extracts the annotation archive if needed and selects either all images or a reproducible seeded subset, reusing existing images. Keep the extracted train annotations: INT8 calibration uses train2017 so no validation image influences calibration.

In [ ]:
if FULL_VAL:
    _download_and_extract(
        "http://images.cocodataset.org/zips/val2017.zip",
        COCO_ROOT,
        is_complete=lambda: sum(path.stat().st_size > 0 for path in (COCO_ROOT / "val2017").glob("*.jpg")) == 5000,
    )

Use the shared selector to ensure every model and runtime sees exactly the same image IDs.

In [ ]:
subset = fetch_coco_val2017(COCO_ROOT, n_images=None if FULL_VAL else N_EVAL, seed=SEED)
print(f"Scoring {len(subset.image_ids)} COCO val2017 images per arm, seed {SEED}.")

## Record provenance and resume completed scores

Record software versions, the installed Git commit when available, the device and both annotation hashes once before scoring. An editable or wheel installation without Git provenance has a null commit; it must not be mistaken for a verified source revision. Use a fresh work directory after changing local source or weights. Resume identifies a row by model, arm, image count, seed, device, RF-DETR version and commit; it does not detect uncommitted source changes.

In [ ]:
installed_versions = {
    dist.metadata["Name"].lower().replace("_", "-"): dist.version for dist in importlib.metadata.distributions()
}
rfdetr_dist = importlib.metadata.distribution("rfdetr")
direct_url = json.loads(rfdetr_dist.read_text("direct_url.json") or "{}")
device = (
    f"cuda: {torch.cuda.get_device_name()}"
    if torch.cuda.is_available()
    else f"mps: {platform.processor()}"
    if torch.backends.mps.is_available()
    else f"cpu: {platform.processor()}"
)
provenance = {
    "rfdetr_version": importlib.metadata.version("rfdetr"),
    "commit": direct_url.get("vcs_info", {}).get("commit_id"),
    "torch_version": torch.__version__,
    "faster_coco_eval_version": installed_versions.get("faster-coco-eval"),
    "rfdetr_plus_version": installed_versions.get("rfdetr-plus"),
    "tensorrt_version": installed_versions.get("tensorrt"),
    "device": device,
    "annotation_sha256": {
        name: hashlib.sha256((COCO_ROOT / "annotations" / name).read_bytes()).hexdigest()
        for name in ("instances_val2017.json", "person_keypoints_val2017.json")
    },
}
print(json.dumps(provenance, indent=2))

Use the same explicit fields for loaded and new rows so changing the scored subset or installed revision cannot silently reuse a different run.

In [ ]:
RESUME_FIELDS = ("model", "arm", "n_images", "seed", "device", "rfdetr_version", "commit")


def resume_key(row: dict[str, Any]) -> tuple[Any, ...]:
    """Identify a score by its model, runtime and evaluation context.

    Args:
        row: A saved result or a proposed scoring context.

    Returns:
        Values of the fields that define a resumable score.
    """
    return tuple(row[field] for field in RESUME_FIELDS)

Load completed JSON lines before constructing models; an interrupted or malformed line fails visibly rather than discarding evidence.

In [ ]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
rows = (
    [json.loads(line) for line in RESULTS_PATH.read_text().splitlines() if line.strip()]
    if RESULTS_PATH.exists()
    else []
)
completed = {resume_key(row) for row in rows}
print(f"Loaded {len(rows)} completed scores from {RESULTS_PATH}.")

## Prepare held-out INT8 calibration

TensorRT INT8 uses the first `N_CALIB` train2017 images sorted by filename, matching Section 8.1 of the CUDA export cookbook's docs-table settings. These images are disjoint from val2017. Existing files are reused. Start a fresh work directory if you change `N_CALIB`; cached engines also depend on calibration settings, package versions and GPU hardware.

In [ ]:
CALIBRATION_DIR = WORK_DIR / "calibration"
train_images = json.loads((COCO_ROOT / "annotations" / "instances_train2017.json").read_text())["images"]
calibration_names = sorted(record["file_name"] for record in train_images)[:N_CALIB]
del train_images
for name in calibration_names:
    destination = CALIBRATION_DIR / name
    if not destination.exists():
        _download("http://images.cocodataset.org/train2017/" + quote(name), destination)
print(f"Calibration: {N_CALIB} train2017 images, first by filename, in {CALIBRATION_DIR}.")

## Build predictors

Each builder returns the same public prediction facade. The PyTorch arm borrows the live eager FP32 model; it does not compile or change preprocessing. TensorRT arms export static batch-1 engines with the keyword sets used by the CUDA docs benchmark. Each model and arm has its own engine directory; a single existing engine is reused. Use a fresh `WORK_DIR` after changing calibration, weights, software or GPU, because these paths alone do not validate an engine's build provenance.

In [ ]:
def build_pytorch_fp32(model: RFDETR, model_name: str) -> RFDETRInference:
    """Wrap the live eager model without compilation or dtype changes.

    Args:
        model: Freshly constructed pretrained FP32 model.
        model_name: Selected checkpoint label, unused by the borrowed runtime.

    Returns:
        A public predictor borrowing the model.
    """
    return RFDETRInference(model)

Keep FP16 engines separate from INT8 so a resumed export cannot load the other precision.

In [ ]:
def build_tensorrt_fp16(model: RFDETR, model_name: str) -> RFDETRInference:
    """Reuse or export one static batch-1 FP16 engine.

    Args:
        model: Pretrained model to export when no engine exists.
        model_name: Checkpoint label for the engine directory.

    Returns:
        A public predictor owning the exported runtime.
    """
    output_dir = WORK_DIR / "engines" / model_name / "tensorrt-fp16"
    engines = sorted(output_dir.glob("*.trt"))
    assert len(engines) <= 1, f"Ambiguous engine cache: {output_dir}"
    if engines:
        engine_path = engines[0]
    else:
        engine_path = model.export(
            format="tensorrt",
            output_dir=str(output_dir),
            batch_size=1,
            verbose=False,
            fp16=True,
        )
    assert not engine_path.name.endswith("_int8.trt"), f"Expected an FP16 engine, got {engine_path}"
    return RFDETRInference(engine_path)

Pass the held-out directory and explicit image limit so INT8 uses all selected calibration images rather than the exporter's default limit.

In [ ]:
def build_tensorrt_int8(model: RFDETR, model_name: str) -> RFDETRInference:
    """Reuse or export one static batch-1 calibrated INT8 engine.

    Args:
        model: Pretrained model to export when no engine exists.
        model_name: Checkpoint label for the engine directory.

    Returns:
        A public predictor owning the exported runtime.
    """
    output_dir = WORK_DIR / "engines" / model_name / "tensorrt-int8"
    engines = sorted(output_dir.glob("*.trt"))
    assert len(engines) <= 1, f"Ambiguous engine cache: {output_dir}"
    if engines:
        engine_path = engines[0]
    else:
        engine_path = model.export(
            format="tensorrt",
            output_dir=str(output_dir),
            batch_size=1,
            verbose=False,
            quantization="int8",
            calibration_data=str(CALIBRATION_DIR),
            max_images=N_CALIB,
        )
    assert engine_path.name.endswith("_int8.trt"), f"Expected an INT8 engine, got {engine_path}"
    return RFDETRInference(engine_path)

## Score and save each completed arm

Keep only one live model in memory and release each predictor before moving to the next arm. The evaluator calls `predict` once per RGB image at batch 1 with default preprocessing. Read the task from the predictor rather than guessing it from the checkpoint name. Append each completed score immediately, including full provenance and elapsed scoring time in seconds; model initialization and engine builds are excluded from that time. A later interruption preserves all completed rows.

In [ ]:
builders = {
    "pytorch-fp32": build_pytorch_fp32,
    "tensorrt-fp16": build_tensorrt_fp16,
    "tensorrt-int8": build_tensorrt_int8,
}
for model_name in tqdm(MODELS, desc="Models"):
    pending_arms = []
    for arm in ARMS:
        context = {**provenance, "model": model_name, "arm": arm, "n_images": len(subset.image_ids), "seed": SEED}
        if resume_key(context) in completed:
            tqdm.write(f"SKIP {model_name} / {arm}: matching resume key already scored.")
        else:
            pending_arms.append(arm)
    if not pending_arms:
        continue
    # The license keyword exists only on the Plus constructors.
    model_class = getattr(rfdetr, model_name)
    model = model_class(accept_platform_model_license=True) if model_name in PLUS_MODELS else model_class()
    for arm in tqdm(pending_arms, desc=model_name, leave=False):
        predictor = builders[arm](model, model_name)
        task = predictor.runtime_info["task"]
        started = time.perf_counter()
        score = evaluate_coco_ap(
            partial(predictor.predict, threshold=0.001, include_source_image=False),
            subset,
            task,
        )
        row = {
            **provenance,
            "model": model_name,
            "arm": arm,
            "seed": SEED,
            "task": task,
            "n_images": score.n_images,
            "wall_seconds": time.perf_counter() - started,
            "box_ap": score.box_ap,
            "box_ap50": score.box_ap50,
            "task_ap": score.task_ap,
            "task_ap50": score.task_ap50,
        }
        with RESULTS_PATH.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row, allow_nan=False) + "\n")
        completed.add(resume_key(row))
        rows.append(row)
        task_summary = "" if score.task_ap is None else f", {task} AP {100 * score.task_ap:.1f}%"
        tqdm.write(
            f"{model_name} / {arm}: box AP {100 * score.box_ap:.1f}%{task_summary}, {score.n_images} images, {row['wall_seconds']:.1f}s"
        )
        del predictor
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Inspect saved results

Reload the append-only file rather than relying on in-memory state, so the displayed DataFrame includes earlier completed runs as well as this session. These raw rows preserve both AP50:95 and AP50 for boxes and each model's additional task. Use the guarded table below for publication: a quick subset check is not a full-val2017 benchmark.

In [ ]:
results = pd.DataFrame(json.loads(line) for line in RESULTS_PATH.read_text().splitlines() if line.strip())

Display every saved row with provenance available for inspection before selecting the current run for the docs table.

In [ ]:
display(results)

## Produce the docs table

Paste the printed Markdown table verbatim into the docs. The 5000-image assertion prevents publishing a quick subset as a full-val2017 result, and the single-device assertion keeps runtime columns comparable. Only the selected models, arms and current resume context are rendered; the saved DataFrame above retains all historical rows. Headline AP is box AP for detection, mask AP for segmentation and keypoint AP for keypoints. Segmentation and keypoint rows also show box AP; detection rows leave that extra column empty. SAB and the PyTorch FP32 delta use the headline metric, with the delta expressed in percentage points.

In [ ]:
# COPIES: docs/learn/benchmarks.md Detection, Segmentation and Keypoints tables (COCO AP50:95). Update both together.
SAB_REFERENCE = {
    "RFDETRAtto": 30.5,
    "RFDETRFemto": 37.8,
    "RFDETRPico": 41.6,
    "RFDETRNano": 48.4,
    "RFDETRSmall": 53.0,
    "RFDETRMedium": 54.7,
    "RFDETRLarge": 56.5,
    "RFDETRXLarge": 58.6,
    "RFDETR2XLarge": 60.1,
    "RFDETRSegNano": 40.3,
    "RFDETRSegSmall": 43.1,
    "RFDETRSegMedium": 45.3,
    "RFDETRSegLarge": 47.1,
    "RFDETRSegXLarge": 48.8,
    "RFDETRSeg2XLarge": 49.9,
    "RFDETRKeypointPreview": 71.8,
}
current_keys = {
    resume_key({**provenance, "model": name, "arm": arm, "n_images": len(subset.image_ids), "seed": SEED})
    for name in MODELS
    for arm in ARMS
}
table_rows = [row for row in results.to_dict("records") if resume_key(row) in current_keys]
assert table_rows, "No results match the current settings."
assert all(row["n_images"] == 5000 for row in table_rows), "Docs tables require full val2017 (5000 images)."
assert len({row["device"] for row in table_rows}) == 1, "Docs tables must use one device."
by_pair = {(row["model"], row["arm"]): row for row in table_rows}
assert len(by_pair) == len(table_rows), "Duplicate scores for the current resume context."
header = ["Model", "Task"]
for arm in ARMS:
    header.extend([f"{arm} AP (%)", f"{arm} box AP (%)"])
header.extend(["SAB", "Δ vs SAB (pytorch-fp32)"])
table_lines = ["| " + " | ".join(header) + " |", "| " + " | ".join(["---"] * len(header)) + " |"]
for name in MODELS:
    available = [by_pair[name, arm] for arm in ARMS if (name, arm) in by_pair]
    if not available:
        continue
    task = available[0]["task"]
    assert all(row["task"] == task for row in available), f"Conflicting tasks for {name}"
    cells = [name, task]
    for arm in ARMS:
        row = by_pair.get((name, arm))
        if row is None:
            cells.extend(["—", "—"])
            continue
        headline_ap = row["box_ap"] if task == "detect" else row["task_ap"]
        cells.extend([f"{100 * headline_ap:.1f}", f"{100 * row['box_ap']:.1f}" if task != "detect" else "—"])
    baseline = by_pair.get((name, "pytorch-fp32"))
    delta = "—"
    if baseline is not None:
        baseline_ap = baseline["box_ap"] if task == "detect" else baseline["task_ap"]
        delta = f"{100 * baseline_ap - SAB_REFERENCE[name]:+.1f}"
    cells.extend([f"{SAB_REFERENCE[name]:.1f}", delta])
    table_lines.append("| " + " | ".join(cells) + " |")
print("\n".join(table_lines))

## Next steps

Compare the full-validation table with the [Benchmarks page](https://rfdetr.roboflow.com/latest/learn/benchmarks/), retaining the harness and device differences alongside any copied results. Continue with the [CUDA export cookbook](../export-cuda/) to measure engine latency and the FP16 versus INT8 accuracy trade-off on your GPU.